# V3 CNO+FM Thetao Plots

Simple notebook for plotting temperature (`thetao`) from `scripts/infer_v3_minimal.py` outputs.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

VARIABLES = ["thetao", "so", "zos", "uo", "vo"]
VAR = "thetao"
CHAN = VARIABLES.index(VAR)

RESULT_DIR = Path("/scratch/emboulaalam/OceanLens_git/results/v3_minimal_day_index_269")

npz_path = sorted(RESULT_DIR.glob("day_*.npz"))[0]
data = np.load(npz_path)
metadata = json.loads((RESULT_DIR / "metadata.json").read_text()) if (RESULT_DIR / "metadata.json").exists() else {}

hr = data["hr"][CHAN]
lr = data["lr"][CHAN]
mu_cno = data["mu_cno"][CHAN]
fm_residual = data["fm_residual"][CHAN]
pred = data["pred_cno_fm"][CHAN]
mask = data["mask"][0] > 0.5
lon = data["lon"] if "lon" in data else np.arange(hr.shape[1])
lat = data["lat"] if "lat" in data else np.arange(hr.shape[0])
extent = [float(np.nanmin(lon)), float(np.nanmax(lon)), float(np.nanmin(lat)), float(np.nanmax(lat))]

print("Loaded:", npz_path)
print("date:", metadata.get("date"))
print("mode:", metadata.get("mode"))
print("ensemble_members:", metadata.get("ensemble_members"))
print("thetao shape:", hr.shape)

In [ ]:
def robust_limits(fields, low=1, high=99):
    values = np.concatenate([x[np.isfinite(x)].ravel() for x in fields])
    return float(np.nanpercentile(values, low)), float(np.nanpercentile(values, high))

def symmetric_limit(fields, percentile=98):
    values = np.concatenate([x[np.isfinite(x)].ravel() for x in fields])
    vmax = float(np.nanpercentile(np.abs(values), percentile))
    return vmax if vmax > 0 else 1.0

def plot_fields(items, cmap="viridis", symmetric=False, unit="degC", title=""):
    fields = [field for _, field in items]
    if symmetric:
        vmax = symmetric_limit(fields)
        vmin = -vmax
    else:
        vmin, vmax = robust_limits(fields)
    fig, axes = plt.subplots(1, len(items), figsize=(5.2 * len(items), 4.2), constrained_layout=True)
    axes = np.atleast_1d(axes)
    image = None
    for ax, (name, field) in zip(axes, items):
        image = ax.imshow(field, origin="lower", extent=extent, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(name)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    fig.colorbar(image, ax=axes, shrink=0.82, label=unit)
    if title:
        fig.suptitle(title)
    return fig

def rmse(a, b):
    return float(np.sqrt(np.mean((a[mask] - b[mask]) ** 2)))

print("RMSE LR vs HR:", rmse(lr, hr))
print("RMSE mu_CNO vs HR:", rmse(mu_cno, hr))
print("RMSE CNO+FM vs HR:", rmse(pred, hr))

## 1. Physical Fields: HR, LR, CNO+FM

In [ ]:
plot_fields(
    [
        ("HR truth", hr),
        ("LR upsampled", lr),
        ("CNO+FM prediction", pred),
    ],
    cmap="viridis",
    symmetric=False,
    unit="thetao (degC)",
    title="Temperature fields",
);

## 2. CNO Correction: HR - LR vs mu_CNO - LR

In [ ]:
plot_fields(
    [
        ("Target correction: HR - LR", hr - lr),
        ("CNO correction: mu_CNO - LR", mu_cno - lr),
    ],
    cmap="coolwarm",
    symmetric=True,
    unit="thetao residual (degC)",
    title="CNO correction compared to total correction",
);

## 3. FM Residual: HR - LR vs FM Residual

In [ ]:
plot_fields(
    [
        ("Target correction: HR - LR", hr - lr),
        ("FM residual", fm_residual),
    ],
    cmap="coolwarm",
    symmetric=True,
    unit="thetao residual (degC)",
    title="FM residual compared to total correction",
);